In [15]:
# Stop Spark Session

spark.stop()

In [2]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Distributed Shared Variables")
    .master("local[*]")
    .config("spark.cores.max", 8)
    .config("spark.executor.cores", 2)
    .config("spark.executor.memory", "512M")
    .getOrCreate()
)

spark


In [5]:
# Read EMP CSV data

_schema = "employee_id int, department_id int, name string, age int, gender string, salary int"
emp = spark.read.format("csv").schema(_schema).option("header", True).load("scratch/input/emp_new.csv")

In [ ]:
emp.show()

In [8]:
# Variable (Lookup)
dept_names = {101 : 'Department 1', 
              102 : 'Department 2', 
              103 : 'Department 3', 
              104 : 'Department 4',
              105 : 'Department 5', 
              106 : 'Department 6', 
              107 : 'Department 7', 
              108 : 'Department 8', 
              109 : 'Department 9', 
              110 : 'Department 10'}


In [9]:
# Broadcast the variable

broadcast_dept_names = spark.sparkContext.broadcast(dept_names)

In [10]:
# Check the value of the variable
broadcast_dept_names.value


{101: 'Department 1',
 102: 'Department 2',
 103: 'Department 3',
 104: 'Department 4',
 105: 'Department 5',
 106: 'Department 6',
 107: 'Department 7',
 108: 'Department 8',
 109: 'Department 9',
 110: 'Department 10'}

In [11]:
# Create UDF to return Department name

from pyspark.sql.functions import udf, col

@udf
def get_dept_names(dept_id):
    return broadcast_dept_names.value.get(dept_id)

In [13]:
emp_final = emp.withColumn("dept_name", get_dept_names(col("department_id")))


In [14]:
emp_final.show(5)

+-----------+-------------+----------+---+------+------+------------+
|employee_id|department_id|      name|age|gender|salary|   dept_name|
+-----------+-------------+----------+---+------+------+------------+
|          1|          101|  John Doe| 30|  Male| 50000|Department 1|
|          2|          101|Jane Smith| 25|Female| 45000|Department 1|
|          3|          102| Bob Brown| 35|  Male| 55000|Department 2|
|          4|          102| Alice Lee| 28|Female| 48000|Department 2|
|          5|          103| Jack Chan| 40|  Male| 60000|Department 3|
+-----------+-------------+----------+---+------+------+------------+
only showing top 5 rows

